# Experiment 3: K-Means Clustering
**Dataset:** Iris (150 samples × 4 features)  
**Objective:** Partition unlabelled data into meaningful clusters, determine optimal k, and visualise results.

---

## 0 · Setup & Imports

In [ ]:
# ── MUST be first matplotlib call so the inline backend is active
# before visualization.py is imported (which would otherwise lock to Agg)
%matplotlib inline

import sys, os

# Resolve src/ relative to *this notebook file*, not the process CWD.
# This makes the import robust regardless of how Jupyter was launched.
_NB_DIR = os.path.dirname(os.path.abspath('__file__'))
_SRC_DIR = os.path.normpath(os.path.join(_NB_DIR, '..', 'src'))
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Project modules
from data_loader        import load_config, load_iris_data, explore_dataset, verify_data_structure
from data_preprocessing import preprocess_pipeline
from optimal_k_finder   import compute_elbow_curve, compute_silhouette_scores, compute_gap_statistic, determine_optimal_k
from clustering_model   import train_kmeans, get_cluster_labels, get_cluster_centers, save_model, run_stability_check
from evaluation         import evaluate_clustering, analyze_cluster_characteristics, generate_clustering_report, save_cluster_labels, save_cluster_centers
from visualization      import (
    plot_elbow_curve, plot_silhouette_scores, plot_gap_statistic,
    visualize_clusters_2d, visualize_clusters_3d,
    plot_feature_pairs, plot_radar_chart
)

print('All imports successful ✓')
print(f'matplotlib backend : {matplotlib.get_backend()}')

In [ ]:
# Resolve config path relative to the notebook, not the CWD
_CFG_PATH = os.path.normpath(os.path.join(_NB_DIR, '..', 'config', 'parameters.json'))

cfg     = load_config(_CFG_PATH)
seed    = cfg['model']['random_seed']
n_init  = cfg['model']['n_init']
max_iter= cfg['model']['max_iter']
tol     = cfg['model']['tol']
k_range = range(cfg['model']['k_range_min'], cfg['model']['k_range_max'] + 1)

print('Configuration loaded:')
for section, values in cfg.items():
    print(f'  [{section}]', values)

## 2 · Data Collection & Exploration

In [ ]:
# Override all data & output paths to be relative to the PROJECT ROOT,
# not the notebook directory or process CWD.
_ROOT = os.path.normpath(os.path.join(_NB_DIR, '..'))

cfg['data']['raw_path']              = os.path.join(_ROOT, 'data', 'raw', 'iris.csv')
cfg['data']['processed_path']        = os.path.join(_ROOT, 'data', 'processed', 'scaled_iris.csv')
cfg['output']['model_path']          = os.path.join(_ROOT, 'models', 'kmeans_model.pkl')
cfg['output']['results_dir']         = os.path.join(_ROOT, 'results') + os.sep
cfg['output']['elbow_plot']          = os.path.join(_ROOT, 'results', 'elbow_plot.png')
cfg['output']['silhouette_plot']     = os.path.join(_ROOT, 'results', 'silhouette_plot.png')
cfg['output']['cluster_2d_plot']     = os.path.join(_ROOT, 'results', 'cluster_visualization_2d.png')
cfg['output']['cluster_3d_plot']     = os.path.join(_ROOT, 'results', 'cluster_visualization_3d.png')
cfg['output']['cluster_centers_csv'] = os.path.join(_ROOT, 'results', 'cluster_centers.csv')
cfg['output']['cluster_labels_csv']  = os.path.join(_ROOT, 'results', 'cluster_labels.csv')
cfg['output']['report_txt']          = os.path.join(_ROOT, 'results', 'clustering_report.txt')

df = load_iris_data(cfg, save_raw=True)
explore_dataset(df)
verify_data_structure(df)

### Quick EDA – Feature Distributions

In [ ]:
feature_cols_eda = [c for c in df.columns if c != 'species']

fig, axes = plt.subplots(1, len(feature_cols_eda), figsize=(16, 4))
for ax, col in zip(axes, feature_cols_eda):
    for sp, color in zip(df['species'].cat.categories,
                         ['#6C63FF', '#FF6584', '#43CBFF']):
        ax.hist(df.loc[df['species'] == sp, col], bins=15,
                alpha=0.65, color=color, label=sp)
    ax.set_title(col.replace('_', ' ').title(), color='#E8E8F0')
    ax.set_facecolor('#1A1D2E')
    ax.tick_params(colors='#E8E8F0')
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Feature Distributions (Raw Data)', color='#E8E8F0', fontsize=13, y=1.01)
fig.patch.set_facecolor('#0F1117')
plt.tight_layout()
plt.show()

## 3 · Preprocessing

In [ ]:
X_scaled, feature_cols, scaler, df_clean = preprocess_pipeline(df, cfg)

## 4 · Optimal-k Discovery

### 4.1 – Elbow Method

In [ ]:
elbow_res = compute_elbow_curve(X_scaled, k_range, seed, n_init, max_iter, tol)

### 4.2 – Silhouette Scores

In [ ]:
sil_res = compute_silhouette_scores(X_scaled, k_range, seed, n_init, max_iter, tol)

### 4.3 – Gap Statistic

In [ ]:
gap_res = compute_gap_statistic(
    X_scaled, k_range,
    n_references=cfg['gap_statistic']['n_references'],
    random_seed=seed, n_init=n_init, max_iter=max_iter, tol=tol,
)

### 4.4 – Aggregate Decision (Majority Vote)

In [ ]:
optimal_k = determine_optimal_k(elbow_res, sil_res, gap_res)
print(f'Optimal k selected: {optimal_k}')

## 5 · Model Training

In [ ]:
km           = train_kmeans(X_scaled, k=optimal_k, random_seed=seed,
                            n_init=n_init, max_iter=max_iter, tol=tol)
labels       = get_cluster_labels(km)
centers_orig = get_cluster_centers(km, scaler, feature_cols)

print('\nCluster centers (original scale):')
display(centers_orig.round(3))

## 6 · Stability Analysis

In [ ]:
stability = run_stability_check(X_scaled, optimal_k, n_runs=5,
                                max_iter=max_iter, tol=tol, n_init=n_init)

## 7 · Evaluation

In [ ]:
metrics         = evaluate_clustering(X_scaled, labels, km.inertia_)
cluster_summary = analyze_cluster_characteristics(df_clean, labels, feature_cols, centers_orig)
display(cluster_summary)

## 8 · Visualisations

All plots are saved to `results/` **and** displayed inline.

### 8.1 – Elbow Curve

In [ ]:
plot_elbow_curve(elbow_res['k_values'], elbow_res['inertias'],
                 optimal_k, cfg['output']['elbow_plot'])
from IPython.display import Image
Image(cfg['output']['elbow_plot'])

### 8.2 – Silhouette Scores

In [ ]:
plot_silhouette_scores(sil_res['k_values'], sil_res['silhouette_scores'],
                       X_scaled, labels, optimal_k, cfg['output']['silhouette_plot'])
from IPython.display import Image
Image(cfg['output']['silhouette_plot'])

### 8.3 – Gap Statistic

In [ ]:
_gap_plot = os.path.join(_ROOT, 'results', 'gap_statistic_plot.png')
plot_gap_statistic(gap_res['k_values'], gap_res['gaps'], gap_res['sdk'],
                   optimal_k, _gap_plot)
from IPython.display import Image
Image(_gap_plot)

### 8.4 – 2-D Cluster Visualisation (PCA)

In [ ]:
visualize_clusters_2d(X_scaled, labels, km.cluster_centers_,
                      feature_cols, optimal_k, cfg['output']['cluster_2d_plot'])
from IPython.display import Image
Image(cfg['output']['cluster_2d_plot'])

### 8.5 – 3-D Cluster Visualisation (PCA)

In [ ]:
visualize_clusters_3d(X_scaled, labels, km.cluster_centers_,
                      optimal_k, cfg['output']['cluster_3d_plot'])
from IPython.display import Image
Image(cfg['output']['cluster_3d_plot'])

### 8.6 – Feature Pair Plot

In [ ]:
_pair_plot = os.path.join(_ROOT, 'results', 'feature_pair_plot.png')
plot_feature_pairs(df_clean, labels, feature_cols, optimal_k, _pair_plot)
from IPython.display import Image
Image(_pair_plot)

### 8.7 – Radar Chart (Cluster Profiles)

In [ ]:
_radar_plot = os.path.join(_ROOT, 'results', 'cluster_radar_chart.png')
plot_radar_chart(centers_orig, feature_cols, optimal_k, _radar_plot)
from IPython.display import Image
Image(_radar_plot)

## 9 · Save Artefacts

In [ ]:
save_model(km, cfg['output']['model_path'])
save_cluster_labels(labels, cfg['output']['cluster_labels_csv'])
save_cluster_centers(centers_orig, cfg['output']['cluster_centers_csv'])

from optimal_k_finder import find_elbow_point
_k_elb = find_elbow_point(elbow_res['inertias'], elbow_res['k_values'])
_k_sil = sil_res['k_values'][int(np.argmax(sil_res['silhouette_scores']))]
_k_gap = gap_res['optimal_k']
rationale = (f'Majority vote — Elbow: k={_k_elb}, Silhouette: k={_k_sil}, '
             f'Gap Statistic: k={_k_gap}. Final: k={optimal_k}')

generate_clustering_report(
    metrics, cluster_summary, centers_orig, stability,
    optimal_k, rationale,
    cfg['output']['report_txt'],
)
print('All artefacts saved.')

## 10 · Summary

In [ ]:
print('=' * 60)
print('  K-MEANS CLUSTERING — RESULTS SUMMARY')
print('=' * 60)
print(f"  Optimal k                   : {optimal_k}")
print(f"  Inertia (WCSS)              : {metrics['inertia']}")
print(f"  Silhouette Score  (max 1.0) : {metrics['silhouette_score']}")
print(f"  Davies-Bouldin    (min 0.0) : {metrics['davies_bouldin_index']}")
print(f"  Calinski-Harabasz           : {metrics['calinski_harabasz_index']}")
print(f"  Inertia stability (std)     : {stability['inertia_std']:.4f}")
print('=' * 60)
print(f"  Results saved to            : {cfg['output']['results_dir']}")
print(f"  Model saved to              : {cfg['output']['model_path']}")
print('=' * 60)